# Embeddings, representaciones semánticas y similitud vectorial
## Fundamentos y laboratorio aplicado

**Asignatura:** Análisis, Procesamiento y Aplicación de Inteligencia Artificial

En sesiones previas se abordaron la definición del problema de IA, las entradas y salidas, la preparación de datos, las métricas y el entrenamiento de modelos. También se experimentó con datos estructurados. Esta sesión extiende esa lógica al texto: antes de que un algoritmo pueda comparar documentos, clasificarlos o recuperar información, se debe decidir **cómo representar numéricamente su contenido**.

**Pregunta guía:** si dos textos expresan ideas parecidas usando palabras diferentes, ¿cómo puede un sistema reconocer esa relación?

Los embeddings convierten unidades de texto en vectores numéricos que pueden capturar relaciones semánticas. La calidad de esa representación depende del modelo, los datos, el idioma, el dominio y la tarea.

# 1. ¿Por qué necesitamos representaciones?

Los algoritmos de aprendizaje automático operan sobre estructuras numéricas. Una oración como “El sistema detecta transacciones bancarias fraudulentas” no es, por sí misma, una matriz de números adecuada para la mayoría de los algoritmos clásicos. Se requiere una transformación.

Una **representación** codifica un objeto en una estructura sobre la cual se pueden realizar operaciones. En procesamiento de lenguaje natural, el objeto puede ser una palabra, un token, una oración, un párrafo o un documento.

## Representar no equivale a comprender
Convertir texto en números no significa que el sistema comprenda como una persona. La codificación recoge propiedades definidas o regularidades aprendidas. Su utilidad debe medirse respecto de una tarea: clasificar mensajes, recuperar documentos, agrupar textos o encontrar pasajes que respondan una consulta.

Considérense:
- A: “El automóvil consume demasiada gasolina”.
- B: “El vehículo tiene un alto consumo de combustible”.
- C: “La receta requiere harina y levadura”.

A y B expresan ideas relacionadas, aunque comparten pocas palabras exactas. C trata otro tema. Un método basado en coincidencia literal puede subestimar la relación entre A y B. Un modelo semántico puede situarlos más cerca, aunque no existe garantía universal de que siempre lo haga.

# 2. Representaciones léxicas frente a representaciones semánticas

## 2.1 Representaciones dispersas: bolsa de palabras y TF-IDF

En una representación dispersa (*sparse*), el vector suele tener muchas dimensiones y la mayoría de sus valores son cero. En bolsa de palabras, cada dimensión corresponde a un término del vocabulario. TF-IDF (Term Frequency - Inverse Document Frequency o Frecuencia de Término - Frecuencia Inversa de Documento) pondera términos considerando su frecuencia en un documento y su capacidad para distinguirlo de otros documentos.

Ejemplo simplificado con vocabulario `["banco", "crédito", "perro", "mascota"]` (conteo de apariciones):

- `"banco crédito"` → `[1, 1, 0, 0]`
- `"crédito bancario"` → `[0, 1, 0, 0]`, porque sin lematización ni *stemming* la palabra "bancario" no coincide con "banco" y queda fuera del vocabulario.

La codificación real depende de la tokenización, el vocabulario y la normalización.

**Ventajas:** relativamente interpretable, eficiente y útil como línea base.  
**Limitaciones:** sinónimos y paráfrasis pueden quedar desconectados; las versiones simples pierden orden y contexto.

## 2.2 Representaciones densas: embeddings

Un embedding es un vector denso de dimensión fija. Sus dimensiones no suelen corresponder directamente a palabras individuales y, por lo general, no tienen una interpretación humana aislada.

Ejemplo ilustrativo, con valores inventados solo para mostrar el formato:

`“automóvil” → [0.21, -0.14, 0.77, 0.09]`  
`“vehículo” → [0.19, -0.11, 0.73, 0.12]`

En un modelo real pueden existir cientos o miles de dimensiones. La cercanía entre vectores puede reflejar relaciones aprendidas durante el entrenamiento.

| Propiedad | TF-IDF | Embeddings densos |
|---|---|---|
| Tipo de vector | Disperso | Denso |
| Dimensiones | Términos del vocabulario | Características aprendidas |
| Sinónimos | Difíciles si no comparten términos | Pueden quedar próximos |
| Interpretabilidad | Relativamente alta por término | Baja para dimensiones individuales |
| Contexto | Limitado en métodos clásicos | Depende de arquitectura y entrenamiento |
| Uso típico | Línea base, clasificación y recuperación léxica | Búsqueda semántica, agrupamiento y recuperación para RAG |

**Idea central:** los embeddings no sustituyen automáticamente a TF-IDF. La elección debe evaluarse para la tarea concreta.

# 3. Definición y tipos de embeddings

Formalmente, un modelo de representación puede expresarse como:

$$
f:\mathcal{X}\rightarrow\mathbb{R}^{d},\qquad f(x)=\mathbf{v}
$$

donde $\mathcal{X}$ es el conjunto de entradas posibles (por ejemplo, textos), $x\in\mathcal{X}$ es una entrada concreta, $f$ es el procedimiento de representación, $\mathbf{v}$ es el vector resultante y $d$ es su dimensión.

Una representación es semánticamente útil cuando las relaciones importantes para la tarea quedan reflejadas en la geometría del espacio. No existe un espacio semántico universal: modelos distintos pueden representar el mismo texto de forma diferente y producir rankings diferentes.

### Tipos de representación
- **Embedding de palabra/token:** representa una palabra o unidad léxica. En modelos contextuales, la representación puede variar según la oración.
- **Embedding de oración:** representa una oración completa, a menudo optimizado para similitud o recuperación.
- **Embedding de documento:** representa un documento o fragmento. Para documentos largos suele ser necesario dividir el contenido en fragmentos (*chunks*).

### Contexto y ambigüedad
En “Depositó dinero en el banco” y “Se sentó en el banco de arena”, la palabra “banco” tiene sentidos distintos. Un vector fijo por palabra puede tener dificultades para reflejar ambos sentidos; una representación contextual puede aprovechar las palabras circundantes.

### ¿Cómo se aprenden?
Según la arquitectura y el objetivo de entrenamiento, el modelo puede aprender a predecir tokens, capturar regularidades contextuales o acercar representaciones de pares de textos relacionados. Los modelos de lenguaje generales no están necesariamente optimizados para comparar oraciones directamente; por eso es importante elegir un modelo de embeddings apropiado para el idioma, dominio y tarea.

# 4. Espacio vectorial: cercanía no significa identidad

Un vector puede entenderse como un punto en un espacio multidimensional. Aunque no podamos visualizar directamente cientos de dimensiones, las operaciones algebraicas siguen siendo válidas. La hipótesis de la similitud semántica es que los vectores cercanos representan unidades relacionadas según el entrenamiento.

No debe confundirse proximidad con equivalencia:
- “El paciente presenta fiebre”.
- “El paciente no presenta fiebre”.

Los textos comparten casi todas las palabras, pero expresan afirmaciones opuestas. Un modelo puede capturar la negación, pero no debe asumirse sin evaluación. También “cómo prevenir una infección” y “cómo diagnosticar una infección” están relacionados por tema, aunque no respondan a la misma intención.

En ámbitos críticos, la cercanía vectorial no sustituye la comprobación explícita de negaciones, cifras, entidades, restricciones y evidencia.


# 5. Medidas de similitud y distancia

## Similitud coseno

$$
\cos(\mathbf{a},\mathbf{b}) = \frac{\mathbf{a}\cdot\mathbf{b}}{\lVert\mathbf{a}\rVert_2\,\lVert\mathbf{b}\rVert_2},
\qquad
\lVert\mathbf{a}\rVert_2=\sqrt{\sum_{i=1}^{d}a_i^{2}}
$$

Es el coseno del ángulo $\theta$ entre los dos vectores: compara su **orientación** y normaliza por sus magnitudes. Para vectores no nulos, el resultado está entre $-1$ y $1$. Con representaciones de componentes no negativas, como TF-IDF, queda entre $0$ y $1$. No hay un umbral universal que convierta el valor en "significado equivalente".

## Producto punto

$$
\mathbf{a}\cdot\mathbf{b}=\sum_{i=1}^{d}a_i b_i=\lVert\mathbf{a}\rVert_2\,\lVert\mathbf{b}\rVert_2\cos\theta
$$

Depende tanto de la orientación como de la magnitud. Algunos modelos se entrenan para utilizar producto punto como puntuación de relevancia.

## Distancia euclidiana

$$
d(\mathbf{a},\mathbf{b})=\lVert\mathbf{a}-\mathbf{b}\rVert_2=\sqrt{\sum_{i=1}^{d}(a_i-b_i)^2}
$$

Mide separación geométrica y es sensible a la magnitud. A diferencia de las dos anteriores, **menor** valor significa **mayor** cercanía.

## Relación entre las medidas

Si los vectores están normalizados ($\lVert\mathbf{a}\rVert_2=\lVert\mathbf{b}\rVert_2=1$), se cumple:

$$
\cos(\mathbf{a},\mathbf{b})=\mathbf{a}\cdot\mathbf{b},
\qquad
\lVert\mathbf{a}-\mathbf{b}\rVert_2^{2}=2-2\cos(\mathbf{a},\mathbf{b})
$$

Por eso, con embeddings normalizados, ordenar por coseno, por producto punto o por distancia euclidiana produce el mismo ranking.

## Ejemplo

Sean $\mathbf{a}=[1,0]$, $\mathbf{b}=[2,0]$ y $\mathbf{c}=[0,1]$. Los vectores $\mathbf{a}$ y $\mathbf{b}$ tienen similitud coseno $1$ porque apuntan en la misma dirección, aunque su distancia euclidiana es $1$ y no $0$.

Para ver que las medidas pueden discrepar, sean además $\mathbf{d}=[10,0]$ y $\mathbf{e}=[1,1]$. Respecto de $\mathbf{a}$: el coseno ordena primero a $\mathbf{d}$ (coseno $1$ frente a $\approx 0.707$), pero la distancia euclidiana ordena primero a $\mathbf{e}$ (distancia $1$ frente a $9$). La métrica debe elegirse según la representación y el objetivo.


## Actividad 1. Verificación geométrica
Antes de ejecutar, anticipar cuál par tendrá la mayor similitud coseno y cuál tendrá la menor distancia euclidiana.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

a = np.array([[1.0, 0.0]])
b = np.array([[2.0, 0.0]])
c = np.array([[0.0, 1.0]])

for nombre, x, y in [("a-b", a, b), ("a-c", a, c), ("b-c", b, c)]:
    coseno = cosine_similarity(x, y)[0, 0]
    producto = (x @ y.T)[0, 0]   # x @ y.T es una matriz 1x1; [0, 0] extrae el escalar
    distancia = euclidean_distances(x, y)[0, 0]
    print(f"{nombre}: coseno={coseno:.3f}, producto punto={producto:.3f}, distancia={distancia:.3f}")

**Preguntas de análisis**
1. ¿Por qué a-b tiene similitud coseno máxima aunque sus coordenadas no son iguales?
2. ¿Qué diferencia aparece entre producto punto y similitud coseno?
3. ¿Por qué una puntuación de similitud no debe interpretarse como probabilidad?

# 6. Laboratorio A: línea base léxica con TF-IDF

TF-IDF pondera términos según su frecuencia en el documento y su capacidad de discriminar entre documentos. Un término común a casi toda la colección suele aportar menos información discriminativa que un término frecuente en un documento, pero raro en el corpus.

$$
\mathrm{TFIDF}(t,d)=\mathrm{TF}(t,d)\times\mathrm{IDF}(t)
$$

La fórmula exacta depende de la implementación. En `TfidfVectorizer` de scikit-learn, con la configuración por defecto, se usa:

$$
\mathrm{TF}(t,d)=f_{t,d},
\qquad
\mathrm{IDF}(t)=\ln\!\left(\frac{1+n}{1+\mathrm{df}(t)}\right)+1
$$

donde $f_{t,d}$ es el número de veces que el término $t$ aparece en el documento $d$, $n$ es el número de documentos del corpus y $\mathrm{df}(t)$ es el número de documentos que contienen a $t$. Después, el vector de cada documento se normaliza para que tenga norma 1:

$$
\mathbf{v}_d=\frac{\mathbf{w}_d}{\lVert\mathbf{w}_d\rVert_2},
\qquad
\mathbf{w}_d=\big[\mathrm{TFIDF}(t,d)\big]_{t\in V}
$$

siendo $V$ el vocabulario. El objetivo es ordenar documentos candidatos según su similitud con una consulta.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

documentos = [
    "El automóvil consume mucha gasolina y requiere mantenimiento frecuente.",
    "El vehículo tiene un consumo elevado de combustible.",
    "La motocicleta necesita combustible y mantenimiento periódico.",
    "La receta de pan requiere harina, agua y levadura.",
    "Un sistema bancario debe detectar transacciones fraudulentas.",
    "La detección de fraude financiero analiza operaciones sospechosas.",
    "El modelo de visión artificial identifica defectos en piezas industriales.",
    "La segmentación de imágenes delimita los píxeles correspondientes a un objeto."
]
consulta = "¿Cómo identificar operaciones financieras sospechosas?"

vectorizador = TfidfVectorizer(lowercase=True, strip_accents="unicode")
matriz_tfidf = vectorizador.fit_transform(documentos)
vector_consulta = vectorizador.transform([consulta])
puntuaciones_tfidf = cosine_similarity(vector_consulta, matriz_tfidf)[0]

resultados_tfidf = pd.DataFrame({
    "documento": documentos,
    "similitud_tfidf": puntuaciones_tfidf
}).sort_values("similitud_tfidf", ascending=False, kind="stable").reset_index(drop=True)

resultados_tfidf

### Interpretación
TF-IDF compara pesos asociados a términos. No infiere de manera general que “operaciones sospechosas” y “transacciones fraudulentas” expresen conceptos cercanos. El resultado depende del vocabulario y de la coincidencia entre términos.

TF-IDF es una línea base, no una referencia absoluta. Un método semántico debe demostrar mejora sobre una tarea y un conjunto de evaluación definidos.

# 7. Laboratorio B: embeddings semánticos de oraciones

Se propone `sentence-transformers`, una biblioteca para obtener vectores a partir de modelos preentrenados. El modelo de referencia es `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, un modelo multilingüe de similitud de oraciones.

La primera ejecución puede descargar pesos desde Internet y requerir varios minutos. Si el entorno no tiene conexión o bloquea descargas, la celda puede fallar; la línea base anterior sigue siendo ejecutable. En proyectos reales deben registrarse modelo, versión, idioma, parámetros y condiciones de ejecución.

In [ ]:
# Si la biblioteca no está instalada, ejecutar:
# %pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

nombre_modelo = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo = SentenceTransformer(nombre_modelo)

embeddings_documentos = modelo.encode(
    documentos, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=False
)
embedding_consulta = modelo.encode(
    [consulta], convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=False
)

puntuaciones_semanticas = cosine_similarity(embedding_consulta, embeddings_documentos)[0]
resultados_embeddings = pd.DataFrame({
    "documento": documentos,
    "similitud_embedding": puntuaciones_semanticas
}).sort_values("similitud_embedding", ascending=False).reset_index(drop=True)

resultados_embeddings

In [ ]:
comparacion = pd.DataFrame({
    "documento": documentos,
    "similitud_tfidf": puntuaciones_tfidf,
    "similitud_embedding": puntuaciones_semanticas
})
comparacion["ranking_tfidf"] = comparacion["similitud_tfidf"].rank(
    ascending=False, method="min"
).astype(int)
comparacion["ranking_embedding"] = comparacion["similitud_embedding"].rank(
    ascending=False, method="min"
).astype(int)
comparacion.sort_values("ranking_embedding")

### Lectura crítica
1. ¿Qué documento ocupa el primer lugar con cada método?
2. ¿Qué relación tienen los resultados mejor posicionados con la intención de la consulta?
3. ¿Qué textos suben por compartir vocabulario y cuáles por expresar conceptos relacionados?
4. ¿La primera posición es relevante? ¿Qué evidencia sustenta la decisión?
5. ¿Qué consultas alternativas permitirían comprobar si el comportamiento se mantiene?

La puntuación no es una probabilidad de corrección ni una medida universal de verdad semántica. Una demostración con una sola consulta tampoco permite concluir que un método sea superior.

# 8. Evaluación de búsqueda semántica

Una evaluación rigurosa requiere consultas representativas, documentos candidatos y juicios de relevancia definidos antes de comparar modelos.

| Consulta | Documento | Relevancia |
|---|---|---:|
| “detectar fraude financiero” | “El sistema identifica transacciones fraudulentas” | 1 |
| “detectar fraude financiero” | “La receta requiere harina y levadura” | 0 |
| “localizar defectos en una pieza” | “El sistema identifica fallos en productos industriales” | 1 |

La relevancia puede ser binaria o graduada. En proyectos reales conviene establecer criterios explícitos y, cuando sea posible, contar con más de una persona evaluadora.

### Métricas frecuentes

Sea $\mathrm{rel}_i\in\{0,1\}$ la relevancia del documento en la posición $i$ del ranking, $k$ el número de resultados considerados y $R$ el número total de documentos relevantes para la consulta.

- **Precision@k:** proporción de los primeros $k$ resultados que son relevantes.

$$
\mathrm{P}@k=\frac{1}{k}\sum_{i=1}^{k}\mathrm{rel}_i
$$

- **Recall@k:** proporción de todos los documentos relevantes que aparecen entre los primeros $k$.

$$
\mathrm{R}@k=\frac{1}{R}\sum_{i=1}^{k}\mathrm{rel}_i
$$

- **MRR** (*Mean Reciprocal Rank*): evalúa qué tan arriba aparece el primer resultado relevante. Sea $Q$ el conjunto de consultas y $\mathrm{rank}_q$ la posición del primer relevante para la consulta $q$ (si no hay ninguno, esa consulta aporta 0).

$$
\mathrm{MRR}=\frac{1}{|Q|}\sum_{q\in Q}\frac{1}{\mathrm{rank}_q}
$$

- **nDCG@k:** evalúa el ordenamiento y puede incorporar relevancia graduada. Se compara el DCG obtenido con el DCG del ordenamiento ideal (IDCG).

$$
\mathrm{DCG}@k=\sum_{i=1}^{k}\frac{\mathrm{rel}_i}{\log_2(i+1)},
\qquad
\mathrm{nDCG}@k=\frac{\mathrm{DCG}@k}{\mathrm{IDCG}@k}
$$

Ejemplo: si tres de los primeros cinco resultados son relevantes, $\mathrm{P}@5 = 3/5 = 0.6$. Esto no significa “60 % de comprensión” ni que la similitud sea una probabilidad.

In [ ]:
import numpy as np

# Ejemplo con etiquetas de relevancia definidas manualmente (posiciones 1 a 5)
relevancia_top5 = [1, 0, 1, 1, 0]
total_relevantes = 4          # R: relevantes que existen en toda la colección para esta consulta
k = 5

precision_at_k = sum(relevancia_top5[:k]) / k
recall_at_k = sum(relevancia_top5[:k]) / total_relevantes
rr = next((1 / (i + 1) for i, r in enumerate(relevancia_top5) if r == 1), 0.0)

dcg = sum(r / np.log2(i + 2) for i, r in enumerate(relevancia_top5[:k]))      # i empieza en 0 -> log2(posición + 1) = log2(i + 2)
ideal = [1] * min(total_relevantes, k) + [0] * (k - min(total_relevantes, k))
idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
ndcg = dcg / idcg

print(f"Precision@{k} = {precision_at_k:.2f}")
print(f"Recall@{k}    = {recall_at_k:.2f}")
print(f"RR (una consulta) = {rr:.2f}")
print(f"nDCG@{k}      = {ndcg:.3f}")

# 9. Limitaciones y riesgos

- **Sesgo del corpus:** las representaciones pueden reflejar asociaciones presentes en los datos de entrenamiento.
- **Dominio e idioma:** un modelo general puede fallar ante terminología especializada, dialectos o lenguaje local.
- **Negación y cifras:** “con alergia” y “sin alergia”, o “5 mg” y “50 mg”, difieren por detalles pequeños pero esenciales.
- **Textos extensos:** los límites de entrada suelen exigir fragmentación; una fragmentación inadecuada puede separar afirmaciones de su contexto.
- **Umbrales:** un valor como 0.70 no tiene significado universal. Debe calibrarse con ejemplos evaluados.
- **Costos operativos:** cálculo, almacenamiento, latencia, privacidad y actualización del índice forman parte del diseño del sistema.

La cercanía vectorial es una señal útil para recuperar candidatos, no una garantía de verdad, equivalencia ni suficiencia.

# 10. Aplicaciones en proyectos universitarios

| Tipo de proyecto | Uso posible | Precaución |
|---|---|---|
| Búsqueda documental | Recuperar documentos relacionados con una consulta | Evaluar relevancia con consultas representativas |
| Clasificación textual | Usar vectores como entrada de un clasificador | Evitar fuga entre entrenamiento y prueba |
| Agrupamiento | Agrupar textos relacionados | Validar la utilidad de los grupos |
| Detección de duplicados | Encontrar registros parecidos | Similitud no implica duplicación real |
| Recomendación | Representar descripciones de contenido | Similitud textual no equivale a preferencia |
| RAG | Recuperar fragmentos para aportar contexto a un LLM | La recuperación incorrecta puede inducir respuestas incorrectas |
| Visión y audio | Usar modelos de embeddings especializados por modalidad | Un embedding textual no sustituye un modelo visual o de audio |

### Conexión con RAG
En generación aumentada por recuperación (RAG), los documentos se fragmentan, se convierten en embeddings y se almacenan en un índice vectorial. Ante una consulta, se calcula su embedding y se recuperan fragmentos cercanos para entregarlos como contexto a un modelo generativo.

La similitud vectorial ayuda a recuperar candidatos; no garantiza que el contenido sea verdadero, suficiente o pertinente. Se requiere evaluar recuperación, conservar trazabilidad de fuentes y verificar las respuestas.

# 11. Actividad integradora

## Escenario
Una institución posee un repositorio de documentos académicos y necesita encontrar material relacionado con consultas de estudiantes, aunque las consultas utilicen expresiones distintas a las de los títulos o resúmenes.

## Trabajo en equipo
1. Definir el problema y qué significa que un documento sea relevante.
2. Proponer al menos seis consultas, incluidas dos con sinónimos o paráfrasis.
3. Ejecutar TF-IDF y embeddings.
4. Registrar los cinco primeros resultados de cada método.
5. Etiquetar la relevancia mediante un criterio explícito.
6. Calcular Precision@5 cuando el conjunto etiquetado lo permita.
7. Analizar un fallo o resultado dudoso.
8. Proponer una mejora y explicar cómo se evaluaría.

| Campo | Registro |
|---|---|
| Problema | Necesidad que resuelve la búsqueda |
| Consulta | Texto utilizado |
| Documento recuperado | Resultado candidato |
| Método | TF-IDF o embeddings |
| Puntuación | Similitud reportada |
| Relevancia | 0 = no relevante; 1 = relevante |
| Posición | Ranking |
| Observación | Acierto, error o limitación |

### Criterios de evaluación
- Comprensión conceptual: 20 %
- Implementación: 20 %
- Evaluación y métrica: 25 %
- Análisis crítico: 25 %
- Reproducibilidad: 10 %

**Entregable:** informe breve con problema, consultas, tabla comparativa, métrica, análisis de un error y recomendación justificada.

# 12. Cierre y pregunta de salida

1. Un embedding transforma una unidad de información en un vector de dimensión fija.
2. La representación resulta útil si conserva relaciones relevantes para una tarea.
3. TF-IDF y los embeddings densos codifican información de maneras distintas.
4. La similitud coseno compara orientación; no es por sí misma una probabilidad.
5. La cercanía semántica debe verificarse con criterios de evaluación apropiados.
6. Una solución rigurosa documenta modelo, métrica, consultas, conjunto de evaluación y limitaciones.

**Pregunta de salida:** ¿Qué evidencia permitiría concluir que un sistema de embeddings recupera información mejor que una línea base TF-IDF para el problema específico?

Una respuesta sólida debe mencionar una tarea definida, consultas representativas, juicios de relevancia, una métrica adecuada, comparación bajo las mismas condiciones y análisis de errores.